# 01 · Çoklu Regresyon: Eğitim Ücreti Ne Kadar Artırır?

Bu derste **Mincer ücret denklemini** tahmin edeceğiz: eğitimin, iş tecrübesinin ve bazı kişisel özelliklerin ücrete etkisi.

**Öğreneceklerin**
1. EKK (En Küçük Kareler) regresyonu kurmak ve sonuç tablosunu okumak
2. Log-düz modelde katsayıları **yüzde değişim** olarak yorumlamak
3. Kare terimle **azalan getiri** (ters U) ilişkisini modellemek
4. Kukla değişkenleri ve **referans grubu** yorumlamak
5. Varsayımları test etmek ve **değişen varyansa** karşı sağlam standart hata kullanmak

> ⚠️ Kullandığımız veri **simüle edilmiştir** (gerçek kişiler değildir). Avantajı: veriyi üreten gerçek modeli biliyoruz, tahminlerimizi "cevap anahtarı" ile karşılaştırabiliriz.

In [1]:
import numpy as np
import ekonometri as ek

veri = ek.veri.yukle("ucret")
veri.head()

,ucret,egitim,tecrube,yas,cinsiyet,bolge,sektor
0,20810,8,5,18,Erkek,Doğu Anadolu,Özel
1,81520,14,9,28,Erkek,Batı Anadolu,Özel
2,154480,12,31,51,Erkek,Batı Anadolu,Özel
3,42340,12,26,45,Erkek,Karadeniz,Kamu
4,53260,12,27,44,Erkek,Orta Anadolu,Özel


Veri setinin değişkenlerine ve onu üreten **gerçek modele** bakalım:

In [2]:
print(ek.veri.aciklama("ucret"))

📁 ucret — Ücret ve eğitim (Mincer denklemi)
⚠️ Simüle veri: gerçek gözlem değildir, eğitim amaçlıdır.

Konu: Çoklu regresyon, log-düz model, kare terim, kukla değişken, değişen varyans

Değişkenler:
  • ucret: Aylık net ücret (TL)
  • egitim: Eğitim süresi (yıl)
  • tecrube: Potansiyel iş tecrübesi (yıl)
  • yas: Yaş
  • cinsiyet: Erkek / Kadın
  • bolge: İstatistiki bölge (7 kategori)
  • sektor: Özel / Kamu

Veriyi üreten gerçek model (cevap anahtarı):
  log(ucret) = 9,55 + 0,085·egitim + 0,036·tecrube − 0,0006·tecrube² − 0,13·Kadın + bölge etkisi + 0,06·Kamu + u;  Var(u) eğitimle artar (değişen varyans).

Örnek:
  ek.ekk("np.log(ucret) ~ egitim + tecrube + I(tecrube**2) + cinsiyet + sektor + bolge", veri)


## 1. Basit bir başlangıç

Önce yalnızca eğitimi kullanalım. Bağımlı değişken olarak ücretin **logaritmasını** alıyoruz; böylece katsayılar yüzde değişim olarak yorumlanır ve ücretlerin sağa çarpık dağılımı dengelenir.

In [3]:
basit = ek.ekk("np.log(ucret) ~ egitim", veri, birimler={"egitim": "yıl"})
basit

Değişken,Katsayı,Std. hata,t,p,
Sabit,"10,0936","0,0429","235,50","<0,0001",***
egitim,"0,0720","0,0033","21,91","<0,0001",***


In [4]:
basit.yorumla()

📊 Yorum — 'ucret' (logaritması)

▸ Model geneli
  • Model, 'ucret' logaritmasındaki değişimin yaklaşık %24,3 kadarını açıklıyor (R² = 0,243, düzeltilmiş R² = 0,242).
  • Açıklayıcı değişkenler birlikte anlamlı (F = 480,14, p < 0,0001): model, yalnızca sabit terimden oluşan modelden daha iyi açıklıyor.
  • Gözlem sayısı: 1.500 · parametre sayısı: 2 · standart hatalar: klasik (homoskedastik).

▸ Katsayılar (diğer değişkenler sabitken)
  • 'egitim' 1 yıl artınca 'ucret' ortalama %7,47 artıyor — %1 düzeyinde istatistiksel olarak anlamlı (p < 0,0001).

▸ Notlar
  • Bağımlı değişken logaritmik: katsayılar yüzde değişim olarak yorumlandı (%100·(eᵇ−1)).
  • Standart hatalar klasik varsayımlara dayanıyor; varsayımları kontrol etmek için: model.varsayimlar()
  • Bu yorumlar ilişkiyi anlatır; nedensellik için araştırma tasarımının (deney, araç değişken vb.) buna izin vermesi gerekir.

## 2. Tam model

Eğitimin etkisi, tecrübe ve diğer özellikler kontrol edilmeden yanlı ölçülebilir (**atlanmış değişken yanlılığı**). Tecrübeyi kare terimiyle ekliyoruz: tecrübe ücreti artırır ama giderek azalan hızla.

Formülde:
- `I(tecrube**2)` → tecrübenin karesi
- `cinsiyet`, `sektor`, `bolge` → metin sütunları otomatik olarak **kukla değişkene** çevrilir; ilk kategori referans gruptur

In [5]:
model = ek.ekk(
    "np.log(ucret) ~ egitim + tecrube + I(tecrube**2) + cinsiyet + sektor + bolge",
    veri,
    birimler={"egitim": "yıl", "tecrube": "yıl"},
)
model

Değişken,Katsayı,Std. hata,t,p,
Sabit,"9,7431","0,0561","173,57","<0,0001",***
cinsiyet: Kadın,"-0,1192","0,0221","-5,39","<0,0001",***
sektor: Kamu,"0,0886","0,0263","3,38","0,0008",***
bolge: Batı Anadolu,"-0,0838","0,0372","-2,25","0,0244",**
bolge: Ege,"-0,1711","0,0392","-4,36","<0,0001",***
bolge: Akdeniz,"-0,2020","0,0388","-5,20","<0,0001",***
bolge: Orta Anadolu,"-0,2736","0,0391","-6,99","<0,0001",***
bolge: Karadeniz,"-0,2431","0,0383","-6,35","<0,0001",***
bolge: Doğu Anadolu,"-0,2693","0,0377","-7,15","<0,0001",***
egitim,"0,0806","0,0031","26,43","<0,0001",***


In [6]:
model.yorumla()

📊 Yorum — 'ucret' (logaritması)

▸ Model geneli
  • Model, 'ucret' logaritmasındaki değişimin yaklaşık %40,3 kadarını açıklıyor (R² = 0,403, düzeltilmiş R² = 0,398).
  • Açıklayıcı değişkenler birlikte anlamlı (F = 91,14, p < 0,0001): model, yalnızca sabit terimden oluşan modelden daha iyi açıklıyor.
  • Gözlem sayısı: 1.500 · parametre sayısı: 12 · standart hatalar: klasik (homoskedastik).

▸ Katsayılar (diğer değişkenler sabitken)
  • 'cinsiyet' değişkeninde 'Kadın' grubunun 'ucret' değeri, referans grupla ('Erkek') karşılaştırıldığında ortalama %11,2 daha düşük — %1 düzeyinde istatistiksel olarak anlamlı (p < 0,0001).
  • 'sektor' değişkeninde 'Kamu' grubunun 'ucret' değeri, referans grupla ('Özel') karşılaştırıldığında ortalama %9,3 daha yüksek — %1 düzeyinde istatistiksel olarak anlamlı (p = 0,0008).
  • 'bolge' değişkeninde 'Batı Anadolu' grubunun 'ucret' değeri, referans grupla ('İstanbul') karşılaştırıldığında ortalama %8,0 daha düşük — %5 düzeyinde istatistiksel olarak anlamlı (p = 0,0244).
  • 'bolge' değişkeninde 'Ege' grubunun 'ucret' değeri, referans grupla ('İstanbul') karşılaştırıldığında ortalama %15,7 daha düşük — %1 düzeyinde istatistiksel olarak anlamlı (p < 0,0001).
  • 'bolge' değişkeninde 'Akdeniz' grubunun 'ucret' değeri, referans grupla ('İstanbul') karşılaştırıldığında ortalama %18,3 daha düşük — %1 düzeyinde istatistiksel olarak anlamlı (p < 0,0001).
  • 'bolge' değişkeninde 'Orta Anadolu' grubunun 'ucret' değeri, referans grupla ('İstanbul') karşılaştırıldığında ortalama %23,9 daha düşük — %1 düzeyinde istatistiksel olarak anlamlı (p < 0,0001).
  • 'bolge' değişkeninde 'Karadeniz' grubunun 'ucret' değeri, referans grupla ('İstanbul') karşılaştırıldığında ortalama %21,6 daha düşük — %1 düzeyinde istatistiksel olarak anlamlı (p < 0,0001).
  • 'bolge' değişkeninde 'Doğu Anadolu' grubunun 'ucret' değeri, referans grupla ('İstanbul') karşılaştırıldığında ortalama %23,6 daha düşük — %1 düzeyinde istatistiksel olarak anlamlı (p < 0,0001).
  • 'egitim' 1 yıl artınca 'ucret' ortalama %8,40 artıyor — %1 düzeyinde istatistiksel olarak anlamlı (p < 0,0001).
  • 'tecrube' değişkeninin etkisi doğrusal değil: önce artan sonra azalan (ters U) bir ilişki var. Etki, 'tecrube' ≈ 29,19 yıl civarında yön değiştiriyor (doğrusal terim p < 0,0001; kare terim p < 0,0001, %1 düzeyinde istatistiksel olarak anlamlı).
  • Ortalama düzeyde ('tecrube' = 22,30 yıl), 'tecrube' 1 yıl artınca 'ucret' yaklaşık %1,0 artıyor.

▸ Notlar
  • Bağımlı değişken logaritmik: katsayılar yüzde değişim olarak yorumlandı (%100·(eᵇ−1)).
  • Standart hatalar klasik varsayımlara dayanıyor; varsayımları kontrol etmek için: model.varsayimlar()
  • Bu yorumlar ilişkiyi anlatır; nedensellik için araştırma tasarımının (deney, araç değişken vb.) buna izin vermesi gerekir.

**Gerçek modelle karşılaştır:** veriyi üreten modelde eğitimin katsayısı 0,085, kadın olmanın etkisi −0,13 idi.
Tahminlerimiz bu değerlere ne kadar yakın? Örneklem büyüdükçe EKK tahmincisi gerçek değere yaklaşır (**tutarlılık**).

In [7]:
model.guven_araliklari().loc[['egitim', 'cinsiyet[T.Kadın]']]

,alt,üst
egitim,0.074650,0.086617
cinsiyet[T.Kadın],-0.162603,-0.075836


## 3. Varsayımlar sağlanıyor mu?

EKK katsayılarının ve özellikle **standart hatalarının** güvenilir olması için bazı varsayımlar gerekir. `model.varsayimlar()` hepsini test eder ve ne yapman gerektiğini söyler.

In [8]:
model.varsayimlar()

,Test,Varsayım,İstatistik,p,Sonuç ve öneri
⚠️,Breusch-Pagan,Sabit varyans,"55,667","<0,0001","p < 0,05: H0 reddedildi — değişen varyans var. Katsayılar yansız ama standart hatalar (ve t/p değerleri) güvenilmez.Öneri: Sağlam standart hata kullan: ek.ekk(..., kovaryans='HC1') — Stata: regress ..., vce(robust)."
⚠️,White,Sabit varyans,"82,304","0,0061","p < 0,05: H0 reddedildi — değişen varyans (ya da model kurgu hatası) var.Öneri: Sağlam standart hata kullan: kovaryans='HC1'. White testi kurgu hatasına da duyarlıdır; RESET testine de bak."
✅,Jarque-Bera,Normal hatalar,"1,122","0,5708","p ≥ 0,05: normallikten belirgin sapma yok."
✅,Ramsey RESET,Doğru fonksiyonel biçim,"0,582","0,6270","p ≥ 0,05: fonksiyonel biçim hatasına dair güçlü kanıt yok."
✅,VIF,Çoklu doğrusal bağlantı yok,"13,280",—,VIF yalnızca bir değişken ile onun karesi arasında yüksek; bu beklenen bir durumdur ve sorun sayılmaz.


Breusch-Pagan testi **değişen varyans** buldu: eğitimli kişilerin ücretleri daha geniş bir aralıkta dağılıyor. Bu durumda katsayılar hâlâ yansızdır ama klasik standart hatalar yanlıştır.

Çözüm: **değişen varyansa dayanıklı (robust) standart hatalar.** Stata'daki `, robust` seçeneğinin karşılığı `kovaryans="HC1"`.

In [9]:
saglam = ek.ekk(
    "np.log(ucret) ~ egitim + tecrube + I(tecrube**2) + cinsiyet + sektor + bolge",
    veri,
    kovaryans="HC1",
    birimler={"egitim": "yıl", "tecrube": "yıl"},
)
saglam

Değişken,Katsayı,Std. hata,t,p,
Sabit,"9,7431","0,0527","184,95","<0,0001",***
cinsiyet: Kadın,"-0,1192","0,0221","-5,39","<0,0001",***
sektor: Kamu,"0,0886","0,0264","3,36","0,0008",***
bolge: Batı Anadolu,"-0,0838","0,0391","-2,15","0,0320",**
bolge: Ege,"-0,1711","0,0374","-4,57","<0,0001",***
bolge: Akdeniz,"-0,2020","0,0394","-5,12","<0,0001",***
bolge: Orta Anadolu,"-0,2736","0,0391","-6,99","<0,0001",***
bolge: Karadeniz,"-0,2431","0,0380","-6,40","<0,0001",***
bolge: Doğu Anadolu,"-0,2693","0,0356","-7,57","<0,0001",***
egitim,"0,0806","0,0029","27,39","<0,0001",***


Katsayılar **aynı**, yalnızca standart hatalar (ve t/p değerleri) değişti. İki modeli yan yana koyalım:

In [10]:
import pandas as pd

pd.DataFrame({
    "katsayı": model.katsayilar,
    "klasik std. hata": model.standart_hatalar,
    "sağlam std. hata (HC1)": saglam.standart_hatalar,
}).round(4)

,katsayı,klasik std. hata,sağlam std. hata (HC1)
Intercept,9.7431,0.0561,0.0527
cinsiyet[T.Kadın],-0.1192,0.0221,0.0221
sektor[T.Kamu],0.0886,0.0263,0.0264
bolge[T.Batı Anadolu],-0.0838,0.0372,0.0391
bolge[T.Ege],-0.1711,0.0392,0.0374
bolge[T.Akdeniz],-0.2020,0.0388,0.0394
bolge[T.Orta Anadolu],-0.2736,0.0391,0.0391
bolge[T.Karadeniz],-0.2431,0.0383,0.0380
bolge[T.Doğu Anadolu],-0.2693,0.0377,0.0356
egitim,0.0806,0.0031,0.0029


## 4. Hipotez testi

"Eğitimin getirisi gerçekten %8,5 mi?" sorusunu **Wald testi** ile sınayalım (H0: katsayı = 0,085):

In [11]:
saglam.wald_testi("egitim = 0.085")

Wald testi — H0: egitim = 0.085
F(1, 1488) = 2,200, p = 0,1383
Sonuç: %5 düzeyinde H0 reddedilemez.

Tecrübe ücreti hiç etkilemiyor mu? (H0: tecrübe ve karesinin katsayıları birlikte sıfır)

In [12]:
saglam.wald_testi(["tecrube", "I(tecrube ** 2)"])

Wald testi — H0: tecrube = 0, I(tecrube ** 2) = 0
F(2, 1488) = 124,745, p < 0,0001
Sonuç: %5 düzeyinde H0 reddedilir.

## 5. Tahmin

16 yıl eğitimli, 10 yıl tecrübeli, İstanbul'da özel sektörde çalışan bir kadının beklenen ücreti:

In [13]:
kisi = pd.DataFrame({"egitim": [16], "tecrube": [10], "cinsiyet": ["Kadın"], "sektor": ["Özel"], "bolge": ["İstanbul"]})
log_tahmin = saglam.tahmin(kisi)
print(f"Beklenen aylık ücret ≈ {np.exp(log_tahmin.iloc[0]):,.0f} TL".replace(",", "."))

Beklenen aylık ücret ≈ 77.699 TL


## 6. Bulgu raporu

`rapor()` Yöntem ve Bulgular metnini tablolarla üretir. Word dosyası için: `saglam.rapor("bulgular.docx")`

> Rapor, modelin hesapladığı sayıları cümleye döker. Metni okuyup kendi çalışmana göre düzelt.

In [14]:
saglam.rapor(etiketler={"ucret": "Aylık ücret", "egitim": "Eğitim süresi", "tecrube": "Tecrübe",
                        "cinsiyet": "Cinsiyet", "sektor": "Sektör", "bolge": "Bölge"})

<Rapor: 'Bulgular', 4 tablo — print(rapor) ya da rapor.kaydet('bulgular.docx')>